# Gene family mapping analysis

This notebook analyzes gene family evolution across bacterial pangenomes using
[PanDelos-plus](https://github.com/synbionics/PanDelos-plus). It compares a reduced
genome subset (Set 1) against the complete dataset (Set 2) to examine how gene families
change when additional genomes are included.

**Prerequisites:** Docker must be installed and running.

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
import json
import subprocess
import matplotlib.pyplot as plt

In [ ]:
# Base directory (notebook location)
notebook_path = os.getcwd()

# Clone PanDelos-plus if not already present
pdp_path = os.path.join(notebook_path, "PanDelos-plus")
if not os.path.exists(pdp_path):
    subprocess.run(["git", "clone",
                     "https://github.com/synbionics/PanDelos-plus.git",
                     pdp_path], check=True)
    print("Repository cloned successfully.")
else:
    print(f"Repository already exists at: {pdp_path}")

# Build Docker image
print("Building Docker image...")
subprocess.run(["docker", "build", "-t", "pandelosplus", "."], cwd=pdp_path, check=True)
print("Docker image built successfully.")

# Setup paths
experiments_path = os.path.join(pdp_path, "files", "pdi")
tool_input_file_path = os.path.join(pdp_path, "input")
tool_output_file_path = os.path.join(pdp_path, "output")
experiment_opath = os.path.join(notebook_path, "output")

for d in [tool_input_file_path, tool_output_file_path, experiment_opath]:
    os.makedirs(d, exist_ok=True)

# Fix permissions on mounted directories to allow Docker container to write
os.chmod(tool_input_file_path, 0o777)
os.chmod(tool_output_file_path, 0o777)

print(f"Input path: {tool_input_file_path}")
print(f"Output path: {tool_output_file_path}")
print(f"Results path: {experiment_opath}")

In [ ]:
# Copy experiment .pdi files to input directory
experiments_files = []
for file in sorted(os.listdir(experiments_path)):
    if file.endswith(".pdi"):
        shutil.copy(os.path.join(experiments_path, file), tool_input_file_path)
        experiments_files.append(file)

print(f"Loaded {len(experiments_files)} experiment files: {experiments_files}")

In [ ]:
# Define genome sets manually.
# set_1: manually specified reduced subset
# set_2: automatically derived from the complete .pdi file

manual_set_1 = {
    "escherichia.pdi": [
        "NC_002695",
        "NC_004431",
        "NC_007946",
    ],
    "mycoplasma.pdi": [
        "NC_000912",
        "NC_004432",
        "NC_007332",
        "NC_009497",
        "NC_011025",
        "NC_013948",
        "NC_014552",
        "NC_014760",
        "NC_014921",
        "NC_015153",
        "NC_016638",
        "NC_017503",
        "NC_017519",
        "NC_018495",
        "NC_018498",
        "NC_019949",
        "NC_021083",
        "NC_021283",
        "NC_022575",
        "NC_022807",
        "NC_023062",
    ],
    "salmonella.pdi": [
        "NC_016832",
        "NZ_LT882486",
    ],
    "xanthomonas.pdi": [
        "NC_003902",
        "NC_003919",
        "NC_013722",
        "NC_020800",
    ],
}

# Only process species with a manual set_1 definition
experiments_files = [f for f in experiments_files if f in manual_set_1]

df = pd.DataFrame(columns=["ifile", "total_number_of_genomes", "set_1_genomes_num",
                            "set_2_genomes_num", "ofile_set_1_genomes_num",
                            "ofile_set_2_genomes_num"])

genomes_sets = {}
for file in experiments_files:
    filepath = os.path.join(tool_input_file_path, file)
    with open(filepath, "r") as f:
        lines = f.readlines()

    # Extract all genomes from the .pdi file for set_2
    all_genomes = set()
    for i in range(0, len(lines), 2):
        genome_id = lines[i].split("\t")[0]
        all_genomes.add(genome_id)

    genomes_sets[file] = {
        "set_1": manual_set_1[file],
        "set_2": sorted(list(all_genomes))
    }

    print(f"File: {file}")
    print(f"  Total genomes: {len(all_genomes)}")
    print(f"  Set 1: {len(genomes_sets[file]['set_1'])} genomes: {genomes_sets[file]['set_1']}")
    print(f"  Set 2 (complete): {len(genomes_sets[file]['set_2'])} genomes")

    new_row = pd.DataFrame({
        "ifile": [file],
        "total_number_of_genomes": [len(all_genomes)],
        "set_1_genomes_num": [len(genomes_sets[file]['set_1'])],
        "set_2_genomes_num": [len(genomes_sets[file]['set_2'])]
    })
    df = pd.concat([df, new_row], ignore_index=True)

In [ ]:
def run_analysis(ifile_path, ifile, genome_ids, set_name):
    """Run PanDelos-plus analysis on a subset of genomes.

    Creates a filtered .pdi file containing only the specified genomes
    and runs PanDelos-plus via Docker.
    """
    print(f"Running analysis on {ifile} ({set_name}, {len(genome_ids)} genomes)")

    # Build output file name (remove .pdi extension to avoid double extension)
    ifile_base = ifile.replace('.pdi', '')
    tmp_file_name = f"{ifile_base}_{set_name}.pdi"
    tmp_file_path = os.path.join(tool_input_file_path, tmp_file_name)

    genome_ids_set = set(genome_ids)

    # Create filtered .pdi file with only the specified genomes
    with open(ifile_path, "r") as f:
        lines = f.readlines()

    lines_to_write = []
    for i in range(0, len(lines), 2):
        genome_id = lines[i].split("\t")[0]
        if genome_id in genome_ids_set:
            lines_to_write.append(lines[i])
            lines_to_write.append(lines[i + 1])

    with open(tmp_file_path, "w") as tmp_f:
        tmp_f.writelines(lines_to_write)

    print(f"  Wrote {len(lines_to_write) // 2} genomes to {tmp_file_name}")

    # Run PanDelos-plus via Docker (paths are relative to the container working dir)
    ofile_name = f"{ifile_base}_{set_name}"
    cmd = [
        "bash", "run-docker.sh",
        "-i", f"input/{tmp_file_name}",
        "-o", f"output/{ofile_name}"
    ]
    cmd_str = ' '.join(cmd)
    print(f"  Command: {cmd_str}")
    # res = subprocess.run(cmd, check=True, shell=True, cwd=pdp_path)
    res = subprocess.run(cmd, check=True, cwd=pdp_path)
    print(f"  PanDelos-plus completed with return code: {res.returncode}")

    print(f"  Completed. Output: {ofile_name}")
    return ofile_name

In [ ]:
# Run PanDelos-plus for each dataset with both genome subsets
for row in df.itertuples():
    ifile = row.ifile
    ifile_path = os.path.join(tool_input_file_path, ifile)

    # Set 1: reduced subset (first 1/3 of genomes)
    genome_ids_set1 = genomes_sets[ifile]["set_1"]
    ofile_set1 = run_analysis(ifile_path, ifile, genome_ids_set1, "set_1")
    df.loc[df.ifile == ifile, "ofile_set_1_genomes_num"] = ofile_set1

    # Set 2: complete dataset (all genomes)
    genome_ids_set2 = genomes_sets[ifile]["set_2"]
    ofile_set2 = run_analysis(ifile_path, ifile, genome_ids_set2, "set_2")
    df.loc[df.ifile == ifile, "ofile_set_2_genomes_num"] = ofile_set2

In [ ]:
def parse_genes(genes_col):
    """Parse the genes column from a JSON string if needed."""
    if isinstance(genes_col, str):
        return json.loads(genes_col)
    return genes_col

def get_gene_ids_from_family(genes_list):
    """Extract the set of gene identifiers from a family."""
    genes = parse_genes(genes_list)
    return set(g['complete-identifier'] for g in genes)

def compute_diffusivity(dataframe):
    """Compute the diffusivity (number of distinct genomes) for each family."""
    diffusivity = {}
    for _, row in dataframe.iterrows():
        family_name = row['family-name']
        genes = parse_genes(row['genes'])
        genomes = set(g['genome-name'] for g in genes)
        diffusivity[family_name] = len(genomes)
    return diffusivity

In [ ]:
# Load JSON results and compute family mappings between set_1 and set_2
all_mappings = []

for row in df.itertuples():
    ifile_base = row.ifile.replace('.pdi', '')
    print(f"\n{'=' * 60}")
    print(f"Processing: {ifile_base}")
    print(f"{'=' * 60}")

    # Load JSON results for both sets
    results_file_set1 = os.path.join(tool_output_file_path, f"{row.ofile_set_1_genomes_num}.json")
    results_file_set2 = os.path.join(tool_output_file_path, f"{row.ofile_set_2_genomes_num}.json")

    with open(results_file_set1, "r") as f:
        df_set1 = pd.DataFrame(json.load(f))
    with open(results_file_set2, "r") as f:
        df_set2 = pd.DataFrame(json.load(f))

    print(f"Set 1: {len(df_set1)} families")
    print(f"Set 2: {len(df_set2)} families")

    # Compute diffusivity for both sets
    diffusivity_set1 = compute_diffusivity(df_set1)
    diffusivity_set2 = compute_diffusivity(df_set2)

    # Pre-compute gene ID sets for each family in set_2
    families_set2 = {}
    for _, r in df_set2.iterrows():
        family_name = r['family-name']
        genes = parse_genes(r['genes'])
        families_set2[family_name] = set(g['complete-identifier'] for g in genes)

    # Map each family in set_1 to the best matching family in set_2
    mapping_data = []
    for _, r in df_set1.iterrows():
        family_set1 = r['family-name']
        genes_f1 = get_gene_ids_from_family(r['genes'])
        size_f1 = len(genes_f1)

        best_match = None
        best_overlap_ratio = 0
        best_match_info = {}

        # Find the best matching family in set_2
        for family_set2_name, genes_f2 in families_set2.items():
            intersection = genes_f1 & genes_f2
            overlap_ratio = len(intersection) / size_f1 if size_f1 > 0 else 0

            if overlap_ratio >= 0.8 and overlap_ratio > best_overlap_ratio:
                best_overlap_ratio = overlap_ratio
                best_match = family_set2_name
                size_f2 = len(genes_f2)

                # Determine match type
                if genes_f1 <= genes_f2:
                    match_type = "unaltered" if size_f1 == size_f2 else "enlarged"
                else:
                    match_type = "partially enlarged"

                best_match_info = {
                    'family_set2': best_match,
                    'genes_set2_count': size_f2,
                    'overlap_count': len(intersection),
                    'overlap_ratio': overlap_ratio,
                    'match_type': match_type,
                    'genes_gained': size_f2 - len(intersection) if match_type != "partially enlarged" else None,
                    'genes_lost': size_f1 - len(intersection) if match_type == "partially enlarged" else 0
                }

        mapping_data.append({
            'species': ifile_base,
            'family_set1': family_set1,
            'genes_set1_count': size_f1,
            'diffusivity_set1': diffusivity_set1.get(family_set1, 0),
            'diffusivity_set2': diffusivity_set2.get(best_match, 0) if best_match else 0,
            **best_match_info
        })

    df_mapping = pd.DataFrame(mapping_data)

    # Mark unmatched families as collapsed
    df_mapping.loc[df_mapping['family_set2'].isna(), 'match_type'] = 'collapsed'

    print(f"\nMapping completed: {len(df_mapping)} families")
    print(f"  Unaltered: {(df_mapping['match_type'] == 'unaltered').sum()}")
    print(f"  Enlarged: {(df_mapping['match_type'] == 'enlarged').sum()}")
    print(f"  Partially enlarged: {(df_mapping['match_type'] == 'partially enlarged').sum()}")
    print(f"  Collapsed: {(df_mapping['match_type'] == 'collapsed').sum()}")

    all_mappings.append(df_mapping)

df_all_mappings = pd.concat(all_mappings, ignore_index=True)
print(f"\nTotal mappings: {len(df_all_mappings)}")

In [ ]:
# Identify families that exist only in set_2 (new families)
print("Identifying families present only in set_2...\n")

new_families_data = []

for row in df.itertuples():
    ifile_base = row.ifile.replace('.pdi', '')

    # Load set_2 results
    results_file_set2 = os.path.join(tool_output_file_path, f"{row.ofile_set_2_genomes_num}.json")
    with open(results_file_set2, "r") as f:
        df_set2 = pd.DataFrame(json.load(f))

    # Find families in set_2 that were not matched from set_1
    matched_set2_families = df_all_mappings[
        (df_all_mappings['species'] == ifile_base) &
        (df_all_mappings['family_set2'].notna())
    ]['family_set2'].unique()

    all_set2_families = df_set2['family-name'].unique()
    unmatched_set2_families = set(all_set2_families) - set(matched_set2_families)

    print(f"{ifile_base}: {len(all_set2_families)} total, "
          f"{len(matched_set2_families)} matched, "
          f"{len(unmatched_set2_families)} new")

    # Compute diffusivity for set_2
    diffusivity_set2 = compute_diffusivity(df_set2)

    # Create entries for new families (diffusivity_set1 = 0)
    for family_name in unmatched_set2_families:
        family_row = df_set2[df_set2['family-name'] == family_name].iloc[0]
        genes = parse_genes(family_row['genes'])

        new_families_data.append({
            'species': ifile_base,
            'family_set1': None,
            'genes_set1_count': 0,
            'diffusivity_set1': 0,
            'diffusivity_set2': diffusivity_set2.get(family_name, 0),
            'family_set2': family_name,
            'genes_set2_count': len(genes),
            'overlap_count': 0,
            'overlap_ratio': 0,
            'match_type': 'new',
            'genes_gained': None,
            'genes_lost': 0
        })

# Extend mappings with new families
df_new_families = pd.DataFrame(new_families_data)
df_all_mappings_extended = pd.concat([df_all_mappings, df_new_families], ignore_index=True)

print(f"\nOriginal mappings: {len(df_all_mappings)}")
print(f"New families added: {len(df_new_families)}")
print(f"Total: {len(df_all_mappings_extended)}")
print(f"\nMatch type distribution:")
for match_type, count in df_all_mappings_extended['match_type'].value_counts().items():
    print(f"  {match_type}: {count}")

In [ ]:
# Species name mapping for display (excluding mycoplasma5)
species_display_names = {
    'escherichia': 'Escherichia coli',
    'salmonella': 'Salmonella enterica',
    'xanthomonas': 'Xanthomonas campestris',
    'mycoplasma': 'Mycoplasma'
}

# Order for table rows (4 organisms only)
species_order = ['escherichia', 'salmonella', 'xanthomonas', 'mycoplasma']

In [ ]:
def create_heatmap_single_organism(species_name, df_species, version_type='shared',
                                   output_path=None, dpi=300, adaptive_size=False,
                                   cell_size=0.6):
    """Create a diffusivity heatmap for a single organism.

    Parameters:
        species_name: Internal species key
        df_species: DataFrame filtered for this species
        version_type: 'shared' (matched families only) or 'all' (includes collapsed and new)
        output_path: Directory to save the output image
        dpi: Output resolution
        adaptive_size: If True, figure size adapts to matrix dimensions.
                       If False, uses fixed (12, 10) size.
        cell_size: Size in inches per cell when adaptive_size=True
    """
    # Filter data based on version type
    size_suffix = "_adaptive" if adaptive_size else ""
    if version_type == 'shared':
        valid_data = df_species[df_species['family_set2'].notna()].copy()
        min_diff_set1 = 1
        min_diff_set2 = 1
        base_filename = f"heatmap_{species_name}_shared{size_suffix}"
    elif version_type == 'all':
        valid_data = df_species.copy()
        valid_data.loc[valid_data['match_type'] == 'collapsed', 'diffusivity_set2'] = 0
        min_diff_set1 = 0
        min_diff_set2 = 0
        base_filename = f"heatmap_{species_name}_all{size_suffix}"
    else:
        raise ValueError("version_type must be 'shared' or 'all'")

    if len(valid_data) == 0:
        print(f"  Warning: No data for {species_name} ({version_type})")
        return

    # Compute count matrix
    max_diff_set1 = int(valid_data['diffusivity_set1'].max())
    max_diff_set2 = int(valid_data['diffusivity_set2'].max())
    rows = max_diff_set2 - min_diff_set2 + 1
    cols = max_diff_set1 - min_diff_set1 + 1
    count_matrix = np.zeros((rows, cols), dtype=int)

    for _, r in valid_data.iterrows():
        d1 = int(r['diffusivity_set1'])
        d2 = int(r['diffusivity_set2'])
        if min_diff_set1 <= d1 <= max_diff_set1 and min_diff_set2 <= d2 <= max_diff_set2:
            count_matrix[d2 - min_diff_set2, d1 - min_diff_set1] += 1

    # Create figure with fixed or adaptive size
    if adaptive_size:
        fig_width = max(4, cols * cell_size + 2)    # +2 for colorbar and labels
        fig_height = max(3, rows * cell_size + 1.5) # +1.5 for title and labels
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    else:
        fig, ax = plt.subplots(figsize=(12, 10))

    max_count = count_matrix.max()
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_under('white')

    im = ax.imshow(count_matrix, aspect='auto', origin='lower', cmap=cmap,
                   vmin=0.5, vmax=max_count, interpolation='none')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Number of families', rotation=270, labelpad=20)

    # Annotate non-zero cells
    for i in range(rows):
        for j in range(cols):
            if count_matrix[i, j] > 0:
                text_color = 'white' if count_matrix[i, j] > max_count / 2 else 'black'
                ax.text(j, i, str(count_matrix[i, j]), ha='center', va='center',
                        fontsize=8, color=text_color, weight='bold')

    # Set axis ticks
    x_step = max(1, cols // 15)
    y_step = max(1, rows // 15)
    ax.set_xticks(range(0, cols, x_step))
    ax.set_xticklabels([min_diff_set1 + i for i in range(0, cols, x_step)], fontsize=10)
    ax.set_yticks(range(0, rows, y_step))
    ax.set_yticklabels([min_diff_set2 + i for i in range(0, rows, y_step)], fontsize=10)
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(-0.5, rows - 0.5)

    ax.set_xlabel('Diffusivity Reduced Set', fontsize=12)
    ax.set_ylabel('Diffusivity Complete Set', fontsize=12)

    display_name = species_display_names.get(species_name, species_name)
    suffix = "(shared families)" if version_type == 'shared' else "(all families)"
    ax.set_title(f'{display_name} - {suffix}', fontsize=14, weight='bold')

    plt.tight_layout(pad=0.6)

    if output_path:
        # Save as PNG
        png_path = os.path.join(output_path, f"{base_filename}.png")
        plt.savefig(png_path, dpi=dpi, bbox_inches='tight', format='png')
        # Save as TIF
        tif_path = os.path.join(output_path, f"{base_filename}.tif")
        plt.savefig(tif_path, dpi=dpi, bbox_inches='tight', format='tif')
        print(f"  Saved: {base_filename}.png and {base_filename}.tif ({valid_data.shape[0]} families)")

    plt.close(fig)
    return base_filename

In [ ]:
# Generate heatmaps for each organism (excluding mycoplasma5)
print("Generating heatmaps...\n")

species_list = [s for s in sorted(df_all_mappings_extended['species'].unique())
                if s != 'mycoplasma5']

for species_name in species_list:
    print(f"{species_name}:")
    df_species = df_all_mappings_extended[
        df_all_mappings_extended['species'] == species_name
    ].copy()

    # Fixed size (12x10)
    create_heatmap_single_organism(
        species_name=species_name, df_species=df_species,
        version_type='shared', output_path=experiment_opath, dpi=300,
        adaptive_size=False)

    create_heatmap_single_organism(
        species_name=species_name, df_species=df_species,
        version_type='all', output_path=experiment_opath, dpi=300,
        adaptive_size=False)

    # Adaptive size
    create_heatmap_single_organism(
        species_name=species_name, df_species=df_species,
        version_type='shared', output_path=experiment_opath, dpi=300,
        adaptive_size=True)

    create_heatmap_single_organism(
        species_name=species_name, df_species=df_species,
        version_type='all', output_path=experiment_opath, dpi=300,
        adaptive_size=True)

print(f"\nHeatmaps saved to: {experiment_opath}")

In [ ]:
# Detailed statistics for each organism
print(f"{'=' * 70}")
print("DETAILED STATISTICS PER ORGANISM")
print(f"{'=' * 70}")

for row in df.itertuples():
    ifile_base = row.ifile.replace('.pdi', '')

    print(f"\n{'_' * 70}")
    print(f"ORGANISM: {ifile_base.upper()}")
    print(f"{'_' * 70}")

    # Load JSON results
    results_file_set1 = os.path.join(tool_output_file_path, f"{row.ofile_set_1_genomes_num}.json")
    results_file_set2 = os.path.join(tool_output_file_path, f"{row.ofile_set_2_genomes_num}.json")

    with open(results_file_set1, "r") as f:
        df_set1 = pd.DataFrame(json.load(f))
    with open(results_file_set2, "r") as f:
        df_set2 = pd.DataFrame(json.load(f))

    # Count total genes for each set
    total_genes_set1 = sum(len(parse_genes(r['genes'])) for _, r in df_set1.iterrows())
    total_genes_set2 = sum(len(parse_genes(r['genes'])) for _, r in df_set2.iterrows())

    print(f"\n  Set 1 (reduced subset):")
    print(f"    Genomes: {row.set_1_genomes_num}")
    print(f"    Families: {len(df_set1)}")
    print(f"    Total genes: {total_genes_set1}")

    print(f"\n  Set 2 (complete dataset):")
    print(f"    Genomes: {row.set_2_genomes_num}")
    print(f"    Families: {len(df_set2)}")
    print(f"    Total genes: {total_genes_set2}")

    # Match statistics
    species_data = df_all_mappings_extended[df_all_mappings_extended['species'] == ifile_base]

    print(f"\n  Family correspondences:")
    print(f"    Families in set_1: {len(df_set1)}")
    print(f"    Families in set_2: {len(df_set2)}")
    shared = species_data['match_type'].isin(['unaltered', 'enlarged', 'partially enlarged']).sum()
    collapsed = (species_data['match_type'] == 'collapsed').sum()
    new = (species_data['match_type'] == 'new').sum()
    print(f"    Shared (matched): {shared}")
    print(f"    Collapsed (set_1 only): {collapsed}")
    print(f"    New (set_2 only): {new}")

    print(f"\n  Match type breakdown:")
    match_counts = species_data['match_type'].value_counts()
    for match_type in ['unaltered', 'enlarged', 'partially enlarged', 'collapsed', 'new']:
        count = match_counts.get(match_type, 0)
        if count > 0:
            print(f"    {match_type.capitalize()}: {count}")

    # Percentages
    total_families_set1 = len(df_set1)
    print(f"\n  Percentages (relative to set_1):")
    print(f"    Matched: {shared}/{total_families_set1} ({100 * shared / total_families_set1:.1f}%)")
    print(f"    Collapsed: {collapsed}/{total_families_set1} ({100 * collapsed / total_families_set1:.1f}%)")

print(f"\n{'=' * 70}")
print("OVERALL SUMMARY")
print(f"{'=' * 70}")
print(f"Total organisms analyzed: {len(df)}")
print(f"Total families mapped: {len(df_all_mappings_extended)}")
print(f"{'=' * 70}")

In [ ]:
# Phase 2: LaTeX Table Generation Functions

def generate_dataset_totals_table(df, df_all_mappings_extended, species_order, species_display_names):
    """
    Generate Table 1: Dataset Overview - Total Counts
    Shows genome counts, gene counts, and family counts for both sets
    """
    
    latex_lines = []
    latex_lines.append(r"\begin{table}[!ht]")
    latex_lines.append(r"    \centering")
    latex_lines.append(r"    \caption{\textbf{Dataset Overview: Total Counts of Genomes, Genes, and Families}}")
    latex_lines.append(r"    \begin{tabularx}{\textwidth}{L Y Y Y Y Y Y}")
    latex_lines.append(r"        \toprule")
    latex_lines.append(r"        Dataset &")
    latex_lines.append(r"        Genomes Set 1 & Genes Set 1 & Families Set 1 &")
    latex_lines.append(r"        Genomes Set 2 & Genes Set 2 & Families Set 2 \\")
    latex_lines.append(r"        \midrule")
    
    # Gather data for each species
    for species_key in species_order:
        display_name = species_display_names[species_key]
        
        # Get data from df
        species_file = f"{species_key}.pdi"
        row_data = df[df['ifile'] == species_file].iloc[0]
        
        genomes_set1 = int(row_data['set_1_genomes_num'])
        genomes_set2 = int(row_data['set_2_genomes_num'])
        
        # Load JSON files to get gene and family counts
        results_file_set1 = os.path.join(tool_output_file_path, f"{row_data['ofile_set_1_genomes_num']}.json")
        results_file_set2 = os.path.join(tool_output_file_path, f"{row_data['ofile_set_2_genomes_num']}.json")
        
        with open(results_file_set1, "r") as f:
            df_set1 = pd.DataFrame(json.load(f))
        
        with open(results_file_set2, "r") as f:
            df_set2 = pd.DataFrame(json.load(f))
        
        # Count genes and families
        genes_set1 = sum(len(parse_genes(r['genes'])) for _, r in df_set1.iterrows())
        genes_set2 = sum(len(parse_genes(r['genes'])) for _, r in df_set2.iterrows())
        families_set1 = len(df_set1)
        families_set2 = len(df_set2)
        
        # Add row to table
        latex_lines.append(f"        {display_name} & {genomes_set1} & {genes_set1} & {families_set1} & {genomes_set2} & {genes_set2} & {families_set2} \\\\")
    
    latex_lines.append(r"        \bottomrule")
    latex_lines.append(r"    \end{tabularx}")
    latex_lines.append(r"    \begin{flushleft}")
    latex_lines.append(r"    Set 1 represents a subsample of genomes used for initial analysis,")
    latex_lines.append(r"    while Set 2 contains the complete dataset. The table shows the total")
    latex_lines.append(r"    number of genomes, genes, and gene families identified in each set")
    latex_lines.append(r"    for four bacterial species groups.")
    latex_lines.append(r"    \end{flushleft}")
    latex_lines.append(r"    \label{tab:dataset_totals}")
    latex_lines.append(r"\end{table}")
    
    return "\n".join(latex_lines)


def generate_correspondence_table(df, df_all_mappings_extended, species_order, species_display_names):
    """
    Generate Table 2: Family Correspondences Between Sets
    Shows shared, collapsed, and new families
    """
    
    latex_lines = []
    latex_lines.append(r"\begin{table}[!ht]")
    latex_lines.append(r"    \centering")
    latex_lines.append(r"    \caption{\textbf{Family Correspondences Between Set 1 and Set 2}}")
    latex_lines.append(r"    \begin{tabularx}{\textwidth}{L Y Y Y Y Y}")
    latex_lines.append(r"        \toprule")
    latex_lines.append(r"        Dataset &")
    latex_lines.append(r"        Families Set 1 & Families Set 2 &")
    latex_lines.append(r"        Shared & Collapsed & New \\")
    latex_lines.append(r"        \midrule")
    
    for species_key in species_order:
        display_name = species_display_names[species_key]
        
        # Get families count from JSON files (same as generate_dataset_totals_table)
        species_file = f"{species_key}.pdi"
        row_data = df[df['ifile'] == species_file].iloc[0]
        
        results_file_set1 = os.path.join(tool_output_file_path, f"{row_data['ofile_set_1_genomes_num']}.json")
        results_file_set2 = os.path.join(tool_output_file_path, f"{row_data['ofile_set_2_genomes_num']}.json")
        
        with open(results_file_set1, "r") as f:
            families_set1 = len(pd.DataFrame(json.load(f)))
        
        with open(results_file_set2, "r") as f:
            families_set2 = len(pd.DataFrame(json.load(f)))
        
        # Count match types from mapping
        species_data = df_all_mappings_extended[df_all_mappings_extended['species'] == species_key]
        shared   = (species_data['match_type'].isin(['unaltered', 'enlarged', 'partially enlarged'])).sum()
        collapsed = (species_data['match_type'] == 'collapsed').sum()
        new      = (species_data['match_type'] == 'new').sum()
        
        latex_lines.append(f"        {display_name} & {families_set1} & {families_set2} & {shared} & {collapsed} & {new} \\\\")
    
    latex_lines.append(r"        \bottomrule")
    latex_lines.append(r"    \end{tabularx}")
    latex_lines.append(r"    \begin{flushleft}")
    latex_lines.append(r"    Families are classified as: Shared (present in both sets with $\geq$80\% gene overlap),")
    latex_lines.append(r"    Collapsed (present in Set 1 but not matched in Set 2), and New (present in Set 2")
    latex_lines.append(r"    but not in Set 1). The correspondence analysis helps identify how gene families")
    latex_lines.append(r"    evolve when additional genomes are added to the dataset.")
    latex_lines.append(r"    \end{flushleft}")
    latex_lines.append(r"    \label{tab:family_correspondence}")
    latex_lines.append(r"\end{table}")
    
    return "\n".join(latex_lines)



def generate_match_types_table(df_all_mappings_extended, species_order, species_display_names):
    """
    Generate Table 3: Match Type Details
    Shows detailed breakdown of match types
    """
    
    latex_lines = []
    latex_lines.append(r"\begin{table}[!ht]")
    latex_lines.append(r"    \centering")
    latex_lines.append(r"    \caption{\textbf{Detailed Classification of Family Match Types}}")
    latex_lines.append(r"    \begin{tabularx}{\textwidth}{L Y Y Y Y Y}")
    latex_lines.append(r"        \toprule")
    latex_lines.append(r"        Dataset &")
    latex_lines.append(r"        Unaltered & Enlarged & Partially Enlarged & Collapsed & New \\")
    latex_lines.append(r"        \midrule")
    
    for species_key in species_order:
        display_name = species_display_names[species_key]
        
        # Filter data for this species
        species_data = df_all_mappings_extended[df_all_mappings_extended['species'] == species_key]
        
        # Count each match type
        match_counts = species_data['match_type'].value_counts()
        unaltered = match_counts.get('unaltered', 0)
        enlarged = match_counts.get('enlarged', 0)
        partially_enlarged = match_counts.get('partially enlarged', 0)
        collapsed = match_counts.get('collapsed', 0)
        new = match_counts.get('new', 0)
        
        # Add row to table
        latex_lines.append(f"        {display_name} & {unaltered} & {enlarged} & {partially_enlarged} & {collapsed} & {new} \\\\")
    
    latex_lines.append(r"        \bottomrule")
    latex_lines.append(r"    \end{tabularx}")
    latex_lines.append(r"    \begin{flushleft}")
    latex_lines.append(r"    Match types are defined as: Unaltered (identical gene content in both sets),")
    latex_lines.append(r"    Enlarged (Set 2 family contains all genes from Set 1 plus additional genes),")
    latex_lines.append(r"    Partially Enlarged (Set 2 family gained genes but also lost some from Set 1),")
    latex_lines.append(r"    Collapsed (no corresponding family found in Set 2), and New (family exists")
    latex_lines.append(r"    only in Set 2, not in Set 1).")
    latex_lines.append(r"    \end{flushleft}")
    latex_lines.append(r"    \label{tab:match_types}")
    latex_lines.append(r"\end{table}")
    
    return "\n".join(latex_lines)

print("LaTeX table generation functions defined successfully.")

In [ ]:
# Generate and save LaTeX tables
print(f"\n{'=' * 70}")
print("GENERATING LATEX TABLES")
print(f"{'=' * 70}\n")

table1_latex = generate_dataset_totals_table(df, df_all_mappings_extended, species_order, species_display_names)
table2_latex = generate_correspondence_table(df, df_all_mappings_extended, species_order, species_display_names)
table3_latex = generate_match_types_table(df_all_mappings_extended, species_order, species_display_names)

# Save tables to files
output_dir = os.path.join(experiment_opath, "latex_tables")
os.makedirs(output_dir, exist_ok=True)

for name, content in [("table1_dataset_totals.tex", table1_latex),
                       ("table2_correspondence.tex", table2_latex),
                       ("table3_match_types.tex", table3_latex)]:
    with open(os.path.join(output_dir, name), "w") as f:
        f.write(content)
    print(f"  Saved: {name}")

print(f"\nAll LaTeX tables saved to: {output_dir}")

In [ ]:
# Bar chart: family count by match type for each species (excluding mycoplasma5)
df_filtered = df_all_mappings_extended[df_all_mappings_extended['species'] != 'mycoplasma5']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = {
    'unaltered': '#2ecc71', 'enlarged': '#3498db',
    'partially enlarged': '#e74c3c', 'collapsed': '#95a5a6', 'new': '#f39c12'
}

for idx, (species, group) in enumerate(df_filtered.groupby('species')):
    ax = axes[idx]
    match_counts = group['match_type'].value_counts()

    ax.bar(match_counts.index, match_counts.values,
           color=[colors.get(x, '#999') for x in match_counts.index])
    ax.set_ylabel('Number of families')

    display_name = species_display_names.get(species, species)
    ax.set_title(f'{display_name}')
    ax.tick_params(axis='x', rotation=45)

    for i, v in enumerate(match_counts.values):
        ax.text(i, v + max(match_counts.values) * 0.02, str(v), ha='center', fontsize=9)

plt.suptitle('Family count by match type', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()